# Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Tốc độ hành lang công nghệ Bay Area 15-60 phút (Caltrans PeMS D4)

**Khung SDG**: SDG 11: "intelligent transportation system (ITS)", "traffic emissions"; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45) theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [ ]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

In [ ]:
# Step 1b: Caltrans PeMS District 4 (Bay Area) station 5-minute files for 2023-2024 and station metadata
import pandas as pd
df = pd.read_parquet(
    "data/processed/pems_d4.parquet"
)
df["ts"] = pd.to_datetime(df["ts"])
meta = pd.read_csv(
    "data/raw/pems/d04_text_meta.txt",
    sep="\t"
)
meta = meta[
    meta["Fwy"].isin([101, 280, 880])
].copy()
h = df.copy()

print("Shape:", df.shape)
print("Stations:", df["station"].nunique())
print("Time:", df["ts"].min(), "→", df["ts"].max())
print("Columns:", df.columns.tolist())

Shape: (7013100, 10)
Stations: 100
Time: 2023-01-01 00:00:00 → 2024-12-31 23:45:00
Columns: ['station', 'ts', 'speed', 'total_flow', 'occupancy', 'Fwy', 'Dir', 'Lanes', 'temp', 'prcp']


### 1.1b. Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan trước khi tự viết EDA ở các ô sau.

In [ ]:
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 442.14it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

In [ ]:
# Step 2: 100 stations on US-101, I-280, I-880 between San Francisco and San Jose, 15-minute aggregate (about 100 thousand rows after sampling)
#df = df[df.station.isin(meta.ID)].dropna(subset=['speed']).drop_duplicates(['station','ts']); df = df[(df.speed > 0) & (df.speed < 100)]
#cnt = df.groupby('station').size(); good = cnt[cnt >= 0.9 * cnt.max()].index[:100]; df = df[df.station.isin(good)]
#h = df.set_index('ts').groupby('station')[['speed','total_flow','occupancy']].resample('15min').mean().reset_index()
#h = h.merge(meta[['ID','Fwy','Dir','Lanes']].rename(columns={'ID':'station'}), on='station', how='left')
#wx = pd.read_csv('data/raw/san_jose_hourly.csv', parse_dates=['ts']); h['ts_h'] = h.ts.dt.floor('h'); h = h.merge(wx.rename(columns={'ts':'ts_h'}), on='ts_h', how='left').drop(columns='ts_h')
#h.to_parquet('data/processed/pems_d4.parquet', index=False); print(h.shape)

(67200, 10)


In [ ]:
# Step 2 — Verify the cleaned dataset

import pandas as pd

h = pd.read_parquet("data/processed/pems_d4.parquet")
h["ts"] = pd.to_datetime(h["ts"])

print("Shape:", h.shape)
print("Number of stations:", h["station"].nunique())
print("Freeways:", sorted(h["Fwy"].dropna().unique()))
print("Minimum speed:", h["speed"].min())
print("Maximum speed:", h["speed"].max())
print("Missing speed:", h["speed"].isna().sum())
print("Time range:", h["ts"].min(), "→", h["ts"].max())

Shape: (7013100, 10)
Number of stations: 100
Freeways: [np.int64(101), np.int64(280), np.int64(880)]
Minimum speed: 3.0
Maximum speed: 82.43333333333334
Missing speed: 0
Time range: 2023-01-01 00:00:00 → 2024-12-31 23:45:00


### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [ ]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
# def clean_log(df0, steps):
#    rows, d = [], df0.copy()
#    for name, fn, why in steps:
#        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
#    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
#_, cleaning_log = clean_log(h, [
#    ('drop duplicates', lambda x: x.drop_duplicates(), 'exact duplicate rows'),
#    ('drop missing target', lambda x: x.dropna(subset=['speed']), 'cannot be forecast'),
#])
#cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)


# Step 2.1 — Cleaning log
# Same logic as Notebook A, implemented with DuckDB for large data.

import duckdb
import pandas as pd

con = duckdb.connect()

raw_path = "data/processed/pems_raw_selected.parquet"
final_path = "data/processed/pems_d4.parquet"

# ---------------------------------------------------------
# 1. Starting dataset
# ---------------------------------------------------------
raw_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{raw_path}')
""").fetchone()[0]

# ---------------------------------------------------------
# 2. Remove missing / invalid speed
#    Notebook A requirement: 0 < speed < 100
# ---------------------------------------------------------
valid_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{raw_path}')
    WHERE speed IS NOT NULL
      AND speed > 0
      AND speed < 100
""").fetchone()[0]

# ---------------------------------------------------------
# 3. Remove duplicate (station, timestamp)
#    Same key used by Notebook A
# ---------------------------------------------------------
valid_unique_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT station, ts
        FROM read_parquet('{raw_path}')
        WHERE speed IS NOT NULL
          AND speed > 0
          AND speed < 100
    )
""").fetchone()[0]

# ---------------------------------------------------------
# 4. Find stations satisfying the 90% completeness rule
# ---------------------------------------------------------
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW station_counts AS
    SELECT
        station,
        COUNT(*) AS n
    FROM (
        SELECT DISTINCT station, ts
        FROM read_parquet('{raw_path}')
        WHERE speed IS NOT NULL
          AND speed > 0
          AND speed < 100
    )
    GROUP BY station
""")

max_station_count = con.execute("""
    SELECT MAX(n)
    FROM station_counts
""").fetchone()[0]

eligible_stations = con.execute(f"""
    SELECT COUNT(*)
    FROM station_counts
    WHERE n >= 0.9 * {max_station_count}
""").fetchone()[0]

# ---------------------------------------------------------
# 5. Select the first 100 eligible stations
#    Same LIMIT 100 logic as Notebook A
# ---------------------------------------------------------
selected_100_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT station
        FROM station_counts
        WHERE n >= 0.9 * {max_station_count}
        ORDER BY station
        LIMIT 100
    ) s
    JOIN (
        SELECT DISTINCT station, ts
        FROM read_parquet('{raw_path}')
        WHERE speed IS NOT NULL
          AND speed > 0
          AND speed < 100
    ) d
    ON s.station = d.station
""").fetchone()[0]

# ---------------------------------------------------------
# 6. 15-minute aggregated dataset
# ---------------------------------------------------------
agg_rows = con.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT *
        FROM read_parquet('data/processed/pems_2023_15min.parquet')

        UNION ALL

        SELECT *
        FROM read_parquet('data/processed/pems_2024_15min.parquet')
    )
""").fetchone()[0]

# ---------------------------------------------------------
# 7. Final dataset after metadata + weather merge
# ---------------------------------------------------------
final_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{final_path}')
""").fetchone()[0]

final_missing_speed = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{final_path}')
    WHERE speed IS NULL
""").fetchone()[0]

# ---------------------------------------------------------
# 8. Create the cleaning log
# ---------------------------------------------------------
cleaning_log = pd.DataFrame([
    [
        "remove missing/invalid speed",
        raw_rows,
        valid_rows,
        raw_rows - valid_rows,
        "Keep observations with 0 < speed < 100"
    ],
    [
        "drop duplicates",
        valid_rows,
        valid_unique_rows,
        valid_rows - valid_unique_rows,
        "Remove duplicate station-timestamp observations"
    ],
    [
        "select 100 stations",
        valid_unique_rows,
        selected_100_rows,
        valid_unique_rows - selected_100_rows,
        "Keep stations with at least 90% of the maximum observation count, then select 100 stations"
    ],
    [
        "5-minute to 15-minute aggregation",
        selected_100_rows,
        agg_rows,
        0,
        "Change temporal resolution from 5 minutes to 15 minutes; row reduction is aggregation, not data deletion"
    ],
    [
        "metadata + weather merge",
        agg_rows,
        final_rows,
        0,
        "Left-merge station metadata and hourly San Jose weather"
    ],
    [
        "final missing-target check",
        final_rows,
        final_rows - final_missing_speed,
        final_missing_speed,
        "Speed is the forecasting target; missing target observations are not used"
    ]
], columns=[
    "step",
    "rows_before",
    "rows_after",
    "dropped",
    "reason"
])

cleaning_log.to_csv(
    "report/table_cleaning_log.csv",
    index=False
)

print("Eligible stations meeting 90% criterion:", eligible_stations)
print("Maximum station observation count:", max_station_count)
print()
display(cleaning_log)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible stations meeting 90% criterion: 1060
Maximum station observation count: 210392



,step,rows_before,rows_after,dropped,reason
0,remove missing/invalid speed,223285760,223285760,0,Keep observations with 0 < speed < 100
1,drop duplicates,223285760,223233820,51940,Remove duplicate station-timestamp observations
2,select 100 stations,223233820,21039200,202194620,Keep stations with at least 90% of the maximum...
3,5-minute to 15-minute aggregation,21039200,7013100,0,Change temporal resolution from 5 minutes to 1...
4,metadata + weather merge,7013100,7013100,0,Left-merge station metadata and hourly San Jos...
5,final missing-target check,7013100,7013100,0,Speed is the forecasting target; missing targe...


### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [ ]:
import pandas as pd
import io
eda = h
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T
num_cols = eda.select_dtypes(include="number").columns                         # numeric statistics
desc_num['skew'] = eda[num_cols].skew()  
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(include=['object', 'category']).describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in num_cols:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False)
print("\nEDA notes:")
display(pd.DataFrame(notes,columns=["column", "note"]))
display(desc_cat)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7013100 entries, 0 to 7013099
Data columns (total 10 columns):
 #   Column      Dtype         
---  ------      -----         
 0   station     int32         
 1   ts          datetime64[us]
 2   speed       float64       
 3   total_flow  float64       
 4   occupancy   float64       
 5   Fwy         int64         
 6   Dir         object        
 7   Lanes       int64         
 8   temp        float64       
 9   prcp        float64       
dtypes: datetime64[us](1), float64(5), int32(1), int64(2), object(1)
memory usage: 508.3+ MB

                count                        mean                  min                  25%                  50%                  75%                  max  \
station     7013100.0                   400193.99             400000.0            400106.75             400200.5            400285.75             400372.0   
ts            7013100  2024-01-01 12:26:50.852546  2023-01-01 00:00:00  2023-07-02 16:00:00  

,column,note
0,speed,"strongly skewed, consider log or tree models"
1,occupancy,"strongly skewed, consider log or tree models"
2,prcp,"strongly skewed, consider log or tree models"


,count,unique,top,freq
Dir,7013100,2,N,3646812


### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [ ]:
# Step 1.3 — EDA part 2: missingness and handling decisions
# RAM-safe implementation — same EDA logic as Notebook A
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv(
    "report/table_missing.csv"
)

print("Missing rate by column:")
display(miss.round(4))

# --------------------------------------------------
# 2. Target missing by time and station
# --------------------------------------------------
tcol, gcol = "ts", "station"

by_time = (
    eda.groupby(
        pd.to_datetime(eda[tcol]).dt.to_period("M")
    )["speed"]
    .apply(lambda s: s.isna().mean())
)

by_unit = (
    eda.groupby(gcol)["speed"]
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
)

print(
    "Target missing by period (top):",
    by_time.sort_values(ascending=False).head(5).round(3).to_dict()
)

print(
    "Target missing by unit (top):",
    by_unit.head(5).round(3).to_dict()
)

# --------------------------------------------------
# 3. Co-missingness matrix
# --------------------------------------------------
# Use int8 instead of int to reduce RAM usage
co = eda.isna().astype("int8")
co = co.loc[:, co.sum() > 0]

if co.shape[1] > 1:
    print("Co-missingness matrix:")
    display(co.corr().round(2))

# --------------------------------------------------
# 4. Missingness figure
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3.3))

miss.missing_rate.head(12).plot.bar(ax=ax)

ax.set_ylabel("Missing rate")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(
    "report/fig_eda_missing.png",
    dpi=300
)
plt.show()
plt.close(fig)

Missing rate by column:


,missing_rate,decision
prcp,0.0409,interpolate/impute median
temp,0.0018,interpolate/impute median
ts,0.0000,keep
station,0.0000,keep
speed,0.0000,keep
total_flow,0.0000,keep
Fwy,0.0000,keep
occupancy,0.0000,keep
Lanes,0.0000,keep
Dir,0.0000,keep


Target missing by period (top): {Period('2023-01', 'M'): 0.0, Period('2023-02', 'M'): 0.0, Period('2023-03', 'M'): 0.0, Period('2023-04', 'M'): 0.0, Period('2023-05', 'M'): 0.0}
Target missing by unit (top): {400000: 0.0, 400001: 0.0, 400002: 0.0, 400006: 0.0, 400007: 0.0}
Co-missingness matrix:


,temp,prcp
temp,1.00,0.18
prcp,0.18,1.00


### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [ ]:
num_cols = [c for c in ['speed', 'total_flow', 'occupancy'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'speed': cols.append([c, 'target', 'keep']); continue
    if c in ('ts', 'station'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'speed'] if 'speed' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('station')['speed'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['ts'] if np.issubdtype(eda['ts'].dtype, np.number) else pd.to_datetime(eda['ts'], errors='coerce').dt.month
by_t = eda.groupby(period)['speed'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

            speed  total_flow  occupancy
speed       1.000      -0.420     -0.861
total_flow -0.420       1.000      0.678
occupancy  -0.861       0.678      1.000
           mean  median     p05     p95        n
station                                         
400319   67.599  68.200  64.167  69.633  70131.0
400214   67.567  68.167  64.067  69.867  70131.0
400350   67.151  69.167  57.550  70.567  70131.0
400093   67.013  68.100  59.200  71.233  70131.0
400138   66.862  67.933  60.133  71.300  70131.0
400237   66.685  67.567  61.767  70.233  70131.0
400322   66.567  67.900  59.933  70.500  70131.0
400318   66.511  66.833  62.933  68.600  70131.0
400160   66.337  67.267  63.133  68.833  70131.0
400231   66.254  67.300  63.167  68.967  70131.0
      mean  median     p05     p95         n
ts                                          
1   62.853  65.167  47.267  70.100  595200.0
2   62.776  65.300  44.633  70.133  547200.0
3   62.696  65.200  43.867  70.067  594400.0
4   63.065  65.367  46.

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [ ]:
# SQL statements run one by one in DuckDB; each SELECT prints its first 20 rows and is saved to report/table_q{k}.csv
import duckdb
if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')   # reuse the connection opened in Step 1
SQL = r"""
-- Step 3: sql/queries.sql
CREATE OR REPLACE TABLE pm AS SELECT * FROM 'data/processed/pems_d4.parquet';
-- Q1: weekly speed profile by freeway and direction; Tuesday to Thursday (office days) versus Monday and Friday
SELECT Fwy, Dir, dayofweek(ts) AS dow, hour(ts) AS hr, AVG(speed) AS speed FROM pm GROUP BY 1, 2, 3, 4 ORDER BY 1, 2, 3, 4;
-- Q2: congestion share (speed below 45 mph) per station, top 20; rain effect
SELECT station, Fwy, AVG((speed < 45)::INT) AS congested_share FROM pm GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 20;
SELECT (prcp > 0) AS rain, AVG((speed < 45)::INT) AS congested_share FROM pm WHERE hour(ts) BETWEEN 7 AND 9 GROUP BY 1;
-- Q3: features and targets 15, 30, 60 minutes ahead
CREATE OR REPLACE TABLE feat AS
SELECT station, Fwy, ts, speed, total_flow, occupancy, Lanes, temp, prcp, hour(ts) AS hr, dayofweek(ts) AS dow,
       LAG(speed, 1) OVER w AS s_lag1, LAG(speed, 4) OVER w AS s_lag4, LAG(speed, 96) OVER w AS s_lag1d, LAG(speed, 672) OVER w AS s_lag1w, AVG(speed) OVER (w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW) AS s_ma1h,
       LAG(total_flow, 1) OVER w AS f_lag1, LEAD(speed, 1) OVER w AS y_15m, LEAD(speed, 2) OVER w AS y_30m, LEAD(speed, 4) OVER w AS y_60m
FROM pm WINDOW w AS (PARTITION BY station ORDER BY ts);
SELECT COUNT(*) AS n, COUNT(y_60m) AS n60 FROM feat;
"""
k = 0
SQL_NO_COMMENTS = '\n'.join(l for l in SQL.splitlines() if not l.strip().startswith('--'))   # drop comment lines first (they may contain semicolons)
for stmt in SQL_NO_COMMENTS.split(';'):
    stmt = stmt.strip()
    if not stmt: continue
    res = con.execute(stmt)
    if stmt.upper().startswith(('SELECT', 'WITH')):
        k += 1; out = res.df(); out.to_csv(f'report/table_q{k}.csv', index=False); print(f'--- Q{k} ---'); print(out.head(20))
open('sql/queries.sql', 'w').write(SQL)

--- Q1 ---
    Fwy Dir  dow  hr      speed
0   101   N    0   0  67.360432
1   101   N    0   1  67.148448
2   101   N    0   2  67.178214
3   101   N    0   3  67.049365
4   101   N    0   4  67.110790
5   101   N    0   5  67.109657
6   101   N    0   6  66.929016
7   101   N    0   7  66.937111
8   101   N    0   8  66.859997
9   101   N    0   9  66.490822
10  101   N    0  10  65.254870
11  101   N    0  11  64.243911
12  101   N    0  12  63.742429
13  101   N    0  13  63.861908
14  101   N    0  14  63.951984
15  101   N    0  15  63.998895
16  101   N    0  16  63.990079
17  101   N    0  17  63.826790
18  101   N    0  18  64.280705
19  101   N    0  19  64.885571
--- Q2 ---
    station  Fwy  congested_share
0    400038  880         0.254524
1    400284  880         0.238240
2    400041  880         0.232907
3    400146  880         0.225592
4    400248  880         0.178338
5    400343  880         0.164278
6    400249  880         0.153712
7    400069  101         0.146996


1309

### 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định

**Làm gì**: từ kết quả Q1, Q2 dựng bảng có số thứ tự đưa thẳng vào bài; kiểm định khác biệt giữa nhóm (Kruskal-Wallis, không cần giả định chuẩn) và tương quan Spearman giữa mục tiêu và các biến ngoại sinh kèm p-value.

**Đọc thế nào**: p < 0,05 nghĩa là khác biệt giữa nhóm khó do ngẫu nhiên; hệ số Spearman cho chiều và độ mạnh quan hệ đơn điệu; kết luận RQ1 phải là một câu có số, ví dụ "mùa đông cao gấp 1,9 lần mùa hè, p < 0,001".

In [ ]:
# 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định
from scipy.stats import kruskal, spearmanr

if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')

# RQ1a: Target by period + Kruskal-Wallis
q1a = con.execute("SELECT EXTRACT(MONTH FROM ts)::INTEGER AS period, speed FROM feat WHERE speed IS NOT NULL").df()
rq1a = q1a.groupby('period')['speed'].agg(['mean','median','std','count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.values for _, s in q1a.groupby('period')['speed'] if len(s) > 30]
if len(groups) > 1: print('Kruskal-Wallis across periods:', kruskal(*groups))
del q1a, groups

# RQ1b: Spearman correlation
schema = con.execute("DESCRIBE feat").df()
numeric_prefixes = ('TINYINT','SMALLINT','INTEGER','BIGINT','UTINYINT','USMALLINT','UINTEGER','UBIGINT','FLOAT','DOUBLE','DECIMAL','HUGEINT')
ext = []
for _, row in schema.iterrows():
    c, dtype = row['column_name'], str(row['column_type']).upper()
    if any(dtype.startswith(prefix) for prefix in numeric_prefixes) and c not in ('speed','y_60m') and not c.startswith('speed'): ext.append(c)
ext = ext[:15]; print('Spearman variables:', ext)

rows = []
for c in ext:
    print(f'Processing Spearman: {c}')
    pair = con.execute(f"SELECT {c}, speed FROM feat WHERE {c} IS NOT NULL AND speed IS NOT NULL").df()
    r, pv = spearmanr(pair[c], pair['speed'], nan_policy='omit') if len(pair) else (np.nan, np.nan)
    rows.append([c, round(r,3) if not pd.isna(r) else np.nan, pv]); del pair

rq1b = pd.DataFrame(rows, columns=['variable','spearman_r','p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print('\nRQ1b — Spearman correlation:'); display(rq1b)

# RQ1c: Target by unit, top and bottom 5
rq1c = con.execute("SELECT station, AVG(speed) AS mean, COUNT(speed) AS count FROM feat WHERE speed IS NOT NULL GROUP BY station ORDER BY mean DESC").df()
rq1c_top_bottom = pd.concat([rq1c.head(5), rq1c.tail(5)])
rq1c_top_bottom.round(3).to_csv('report/table_rq1c_units.csv'); print('\nRQ1c — top and bottom 5 stations:'); display(rq1c_top_bottom.round(3))

# RQ1 conclusion
highest_period, highest_mean, lowest_mean = rq1c_top_bottom.station.iloc[0] if False else rq1a['mean'].idxmax(), rq1a['mean'].max(), rq1a['mean'].min()
ratio = highest_mean / max(lowest_mean, 1e-9)
print('RQ1 CONCLUSION (fill with real numbers): highest period', highest_period, 'is', round(ratio,2), 'times the lowest period')

          mean  median    std   count
period                               
1       62.853  65.167  8.566  595200
2       62.776  65.300  8.962  547200
3       62.696  65.200  9.020  594400
4       63.065  65.367  8.616  576000
5       62.540  65.300  9.430  595200
6       62.566  65.067  8.831  576000
7       62.672  65.033  8.579  595200
8       62.263  65.000  9.414  595200
9       62.843  65.500  9.181  576000
10      62.332  65.100  9.642  595200
11      62.259  65.067  9.537  576000
12      62.652  65.067  8.863  591500
Kruskal-Wallis across periods: KruskalResult(statistic=np.float64(7013.907554473719), pvalue=np.float64(0.0))
Spearman variables: ['station', 'Fwy', 'total_flow', 'occupancy', 'Lanes', 'temp', 'prcp', 'hr', 'dow', 's_lag1', 's_lag4', 's_lag1d', 's_lag1w', 's_ma1h', 'f_lag1']
Processing Spearman: station
Processing Spearman: Fwy
Processing Spearman: total_flow
Processing Spearman: occupancy
Processing Spearman: Lanes
Processing Spearman: temp
Processing Spearman: p

,variable,spearman_r,p_value
13,s_ma1h,0.957,0.000000e+00
9,s_lag1,0.948,0.000000e+00
10,s_lag4,0.835,0.000000e+00
12,s_lag1w,0.785,0.000000e+00
11,s_lag1d,0.743,0.000000e+00
3,occupancy,-0.721,0.000000e+00
14,f_lag1,-0.569,0.000000e+00
2,total_flow,-0.568,0.000000e+00
5,temp,-0.291,0.000000e+00
7,hr,-0.115,0.000000e+00



RQ1c — top and bottom 5 stations:


,station,mean,count
0,400319,67.599,70131
1,400214,67.567,70131
2,400350,67.151,70131
3,400093,67.013,70131
4,400138,66.862,70131
95,400146,56.792,70131
96,400172,56.518,70131
97,400038,55.615,70131
98,400041,55.143,70131
99,400284,54.167,70131


RQ1 CONCLUSION (fill with real numbers): highest period 4 is 1.01 times the lowest period


## Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

Mỗi hình gắn với một ý của RQ1 và có caption là kết luận. Bộ hình: (1) phân phối mục tiêu, histogram và boxplot; (2) chuỗi theo thời gian của vài đơn vị; (3) heatmap thời kỳ x đơn vị; (4) boxplot mục tiêu theo thời kỳ; (5) scatter mục tiêu với biến ngoại sinh mạnh nhất; (6) heatmap tương quan; (7) so sánh nhóm bằng cột có số. Quy tắc: chữ tiếng Anh, bỏ viền trên và phải, số in đậm trên cột, 300 dpi, tên file `fig_rq1_*.png`.

In [ ]:
import seaborn as sns
def style(ax): ax.spines[['top', 'right']].set_visible(False)
d = q1.dropna(subset=['speed'])
# (1) distribution
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['speed'], bins=50, color='#1B6B6D'); ax[0].set_xlabel('speed '); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['speed'], vert=False); ax[1].set_xlabel('speed'); style(ax[1])
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)
# (2) time series of the first 3 units
units = d['station'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['station'] == u].sort_values('ts'); ax.plot(s['ts'], s['speed'], lw=0.8, label=str(u))
ax.set_ylabel('speed'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)
# (3) heatmap period x unit (top 15 units)
top = d.groupby('station')['speed'].mean().nlargest(15).index
hm = d[d['station'].isin(top)].pivot_table(index='station', columns='period', values='speed', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4)); sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'speed'}); ax.set_xlabel('Period'); ax.set_ylabel('Unit')
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)
# (4) boxplot by period
fig, ax = plt.subplots(figsize=(9, 3)); sns.boxplot(data=d, x='period', y='speed', color='#9FBFBF', ax=ax, showfliers=False); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)
# (5) scatter against the strongest exogenous variable
if len(rq1b):
    v = rq1b.iloc[0].variable
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.scatter(d[v], d['speed'], s=4, alpha=0.3, color='#1B6B6D'); ax.set_xlabel(v); ax.set_ylabel('speed'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)
# (6) correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(d.select_dtypes('number').corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)
# (7) group comparison with labelled bars
gm = d.groupby('period')['speed'].mean()
fig, ax = plt.subplots(figsize=(8, 3)); bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D'); ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8); ax.set_ylabel('Mean speed'); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)
print('7 figures saved to report/. Write each caption as: conclusion + number + condition.')

7 figures saved to report/. Write each caption as: conclusion + number + condition.


## Test case của Notebook_A (chạy trước khi bàn giao)

Mỗi assert là một điều kiện phải đúng; nếu sai, sửa bước tương ứng rồi chạy lại. Sau khi qua hết, ô cuối tạo `manifest.json` để Sinh viên B kiểm tra.

In [ ]:
import hashlib, json, os
feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['y_60m'].notna().sum() > 0.8 * len(feat), 'too many missing targets'
key = ['station', 'ts']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'
lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'y_60m']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {'rows': int(len(feat)), 'cols': list(feat.columns), 'md5': h,
            'time_min': str(feat['ts'].min()), 'time_max': str(feat['ts'].max()), 'author': 'Student A'}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str); print(manifest)

Tests A: OK
{'rows': 7013100, 'cols': ['station', 'Fwy', 'ts', 'speed', 'total_flow', 'occupancy', 'Lanes', 'temp', 'prcp', 'hr', 'dow', 's_lag1', 's_lag4', 's_lag1d', 's_lag1w', 's_ma1h', 'f_lag1', 'y_15m', 'y_30m', 'y_60m'], 'md5': '6dba4fed8bd89330920b8812fb74764f', 'time_min': '2023-01-01 00:00:00', 'time_max': '2024-12-31 23:45:00', 'author': 'Student A'}


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [ ] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [ ] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [ ] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [ ] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [ ] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [ ] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [ ] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [ ] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [ ] Audit Log cá nhân có đủ prompt của tuần
- [ ] Báo cáo năm mục đã dán vào kênh nhóm trước 12:00 thứ Tư hoặc thứ Bảy


## Mẫu báo cáo tiến độ (slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45), 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................